First of all, I'm trying to think which version to use first and what changes to make to it!
what I think I'll do for the first submission is: Partial fine tuning (and better cleaning) the v2 phase 1 submission
Viterbi decoding

bidir rnn
# Citations: 
1) Training-free layer selection for partial fine-tuning of language
models
2) https://ocw.mit.edu/courses/6-02-introduction-to-eecs-ii-digital-communication-systems-fall-2012/resources/lecture-7-viterbi-decoding/


In [ ]:
# here I'm trying to refine submission 2

In [ ]:
!pip install ijson
import os
import sys
import json
import math
import time
import random
import zipfile
from typing import List, Tuple, Dict, Any, Generator
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from tqdm.auto import tqdm
import numpy as np, ijson
from multiprocessing import Pool
import regex

In [ ]:
import torch
ck = torch.load("/kaggle/input/datasets/koooshiieee/phase2v2/model.pt", map_location="cpu", weights_only=False)
print(type(ck), list(ck.keys())[:10] if isinstance(ck, dict) else "")
print(ck.get("cfg") or ck.get("config") if isinstance(ck, dict) else "")

In [ ]:

HF_TOKEN = None
if "HF_TOKEN" in os.environ:
    HF_TOKEN = os.environ["HF_TOKEN"]
else:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        HF_TOKEN = user_secrets.get_secret("HF")
        os.environ["HF_TOKEN"] = HF_TOKEN
        print("Successfully loaded HF_TOKEN from Kaggle Secrets.")
    except Exception as e:
        print(f"Note: Kaggle secrets not detected or HF key missing ({e}). Will attempt public access.")

if HF_TOKEN:
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN)
        print("Huggingface hub authentication successful.")
    except Exception as err:
        print(f"HF Login warning: {err}")


print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
from huggingface_hub import HfApi

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

HF_TOKENIZER_UPLOAD_REPO = os.environ.get("HF_TOKENIZER_UPLOAD_REPO", "legal-bpe-tokenizer-phase1")



TRAIN_FILES = ["Phase_2_LNER/Train/fold1-gold.json", "Phase_2_LNER/Train/fold1-text.json",
              "Phase_2_LNER/Train/fold2-text.json", "Phase_2_LNER/Train/fold2-gold.json"]

TEST_FILES = ["Phase_2_LNER/Test/fold3-text.json"]

VOCAB_SIZE = 33000
SEED = 67
random.seed(SEED)
torch.manual_seed(SEED)

KNOWN_COUNTS = {"train_supreme_court.json": 55549, "train_acts.json": 18826,
                "train_high_courts.json": 2536738, "train_tribunals.json": 869132}

HF_TOK_REPO = f"{HfApi(token=HF_TOKEN).whoami()['name']}/legal-bpe-tokenizer-phase2-v1"

OUTPUT_DIR = "/kaggle/working/cs781_phase2" if os.path.exists("/kaggle") else "./outputs"
TOKENIZER_DIR = os.path.join(OUTPUT_DIR, "custom_bpe_tokenizer_v2")
TOKEN_DATA_DIR = os.path.join(OUTPUT_DIR, "token_data_v3")

CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "model_checkpoint")
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")
LOSS_PLOT_PATH = os.path.join(OUTPUT_DIR, "training_validation_loss.png")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TOKENIZER_DIR, exist_ok=True)
os.makedirs(TOKEN_DATA_DIR, exist_ok=True)


TRAIN_BIN = os.path.join(TOKEN_DATA_DIR, "train.bin")
VAL_BIN   = os.path.join(TOKEN_DATA_DIR, "val.bin")


In [ ]:

from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, processors
from transformers import PreTrainedTokenizerFast
from huggingface_hub import hf_hub_download

def download_data():
    local_train_paths = []
    print("\nDownloading phase 2 training data.")
    for tf in TRAIN_FILES:
        try:
            p = hf_hub_download(repo_id = HF_DATASET_REPO, filename = tf, repo_type = "dataset", token = HF_TOKEN)
            local_train_paths.append(p)
            print(f"Downloaded: {tf}")
        except Exception as e:
            print(f"Warning downloading {tf}: {e}")
            
    test_path = None
    for tf in TEST_FILES:
        try:
            test_path = hf_hub_download(repo_id = HF_DATASET_REPO, filename = tf, repo_type = "dataset", token = HF_TOKEN)
            print(f"Downloaded Test File: {tf}")
            break
        except Exception:
            continue
            
    return local_train_paths, test_path

In [ ]:
train_paths, test_path = download_data()

# inspect_json_files(train_paths, n=2)

In [ ]:
P = {os.path.basename(p): p for p in train_paths}
TEST = test_path
print(P.keys(), TEST)

In [ ]:
%%writefile bpe_tokenizer.py
#!/usr/bin/env python3
import os, json, regex
from typing import Dict, List, Tuple
from collections import Counter, defaultdict
from tqdm.auto import tqdm

import regex


_PAT = regex.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")


class PureByteLevelBPETokenizer:
    def __init__(self, vocab_size: int = 33000):
        self.vocab_size = vocab_size
        self.special_tokens = ["<pad>", "<unk>", "<s>", "</s>", "<mask>"]

        self.encoder: Dict[bytes, int] = {}
        self.decoder: Dict[int, bytes] = {}
        self.merges: Dict[Tuple[bytes, bytes], bytes] = {}       # pair -> merged token
        self.merge_ranks: Dict[Tuple[bytes, bytes], int] = {}    # pair -> order learned (lower = earlier)
        self._cache: Dict[str, List[int]] = {}

        self._init_base_vocab()

    def _init_base_vocab(self):
        for b in range(256):
            b_bytes = bytes([b])
            self.encoder[b_bytes] = b
            self.decoder[b] = b_bytes

        # special tokens
        for idx, token in enumerate(self.special_tokens, start=256):
            b_bytes = token.encode("utf-8")
            self.encoder[b_bytes] = idx
            self.decoder[idx] = b_bytes

    @property
    def pad_token_id(self) -> int:
        return self.encoder["<pad>".encode("utf-8")]

    @property
    def unk_token_id(self) -> int:
        return self.encoder["<unk>".encode("utf-8")]

    @property
    def bos_token_id(self) -> int:
        return self.encoder["<s>".encode("utf-8")]

    @property
    def eos_token_id(self) -> int:
        return self.encoder["</s>".encode("utf-8")]

    # ------------------------------------------------------------------ training
    @staticmethod
    def _merge_word(word, a, b, new):
        out, i, n = [], 0, len(word)
        while i < n:
            if i < n - 1 and word[i] == a and word[i + 1] == b:
                out.append(new)
                i += 2
            else:
                out.append(word[i])
                i += 1
        return out

    def train_from_texts(self, texts: List[str], target_vocab_size: int = None, min_word_freq: int = 2):
        if target_vocab_size is None:
            target_vocab_size = self.vocab_size
        num_merges = target_vocab_size - len(self.encoder)
        print(f"\nTraining byte level BPE from scratch. Target merges: {num_merges:,} | Base Vocab: {len(self.encoder)}")

        # 1) pre-tokenize + count unique words (the BPE then runs on this small table, not the raw text)
        word_counts = Counter()
        for text in tqdm(texts, desc="Counting pre-tokens"):
            if text and text.strip():
                word_counts.update(_PAT.findall(text))

        words, freqs = [], []
        for w, c in word_counts.items():
            if c >= min_word_freq:
                words.append([bytes([b]) for b in w.encode("utf-8", "replace")])
                freqs.append(c)
        del word_counts
        print(f"Unique words used for merge learning: {len(words):,}")

        # 2) initial pair statistics (updated incrementally afterwards)
        pair_counts = defaultdict(int)
        where = defaultdict(set)
        for i, w in enumerate(tqdm(words, desc="Initial pair counts")):
            for p in zip(w, w[1:]):
                pair_counts[p] += freqs[i]
                where[p].add(i)

        # 3) merge loop
        pbar = tqdm(total=num_merges, desc="Learning BPE Merges", dynamic_ncols=True)
        for _ in range(num_merges):
            if not pair_counts:
                print("No more candidate pairs found to merge.")
                break

            best_pair, freq = max(pair_counts.items(), key=lambda kv: kv[1])
            if freq < 2:
                print(f"Stopping early: Most frequent pair frequency is {freq}")
                break

            a, b = best_pair
            merged_token = a + b
            self.merges[best_pair] = merged_token
            self.merge_ranks[best_pair] = len(self.merge_ranks)
            if merged_token not in self.encoder:          # two merge paths can give the same bytes
                new_id = len(self.encoder)
                self.encoder[merged_token] = new_id
                self.decoder[new_id] = merged_token

            for i in list(where[best_pair]):
                w, f = words[i], freqs[i]
                for p in zip(w, w[1:]):
                    pair_counts[p] -= f
                    if pair_counts[p] <= 0:
                        del pair_counts[p]
                w = self._merge_word(w, a, b, merged_token)
                words[i] = w
                for p in zip(w, w[1:]):
                    pair_counts[p] += f
                    where[p].add(i)
            del where[best_pair]
            pair_counts.pop(best_pair, None)

            pbar.update(1)
            pbar.set_postfix({"vocab_size": len(self.encoder), "top_pair_freq": freq})

        pbar.close()
        self._cache.clear()
        print(f"Training finished! Final Vocab Size: {len(self.encoder):,}")

    # ------------------------------------------------------------------ encode / decode
    def _encode_word(self, word: str) -> List[int]:
        cached = self._cache.get(word)
        if cached is not None:
            return cached
        tokens = [bytes([b]) for b in word.encode("utf-8", "replace")]
        while len(tokens) >= 2:
            best_pair, best_rank = None, None
            for p in zip(tokens, tokens[1:]):
                r = self.merge_ranks.get(p)
                if r is not None and (best_rank is None or r < best_rank):
                    best_pair, best_rank = p, r
            if best_pair is None:
                break
            tokens = self._merge_word(tokens, best_pair[0], best_pair[1], self.merges[best_pair])
        ids = [self.encoder.get(t, self.unk_token_id) for t in tokens]
        if len(self._cache) < 1_000_000:
            self._cache[word] = ids
        return ids

    def encode(self, text: str) -> List[int]:
        ids: List[int] = []
        for word in _PAT.findall(text):
            ids.extend(self._encode_word(word))
        return ids

    def decode(self, token_ids: List[int]) -> str:
        raw_bytes = b"".join(self.decoder.get(i, b"<unk>") for i in token_ids)
        return raw_bytes.decode("utf-8", errors="replace")

    # ------------------------------------------------------------------ save / load
    def save(self, save_dir: str):
        os.makedirs(save_dir, exist_ok=True)
        vocab_path = os.path.join(save_dir, "vocab.json")
        merges_path = os.path.join(save_dir, "merges.json")
        config_path = os.path.join(save_dir, "tokenizer_config.json")

        # latin1 maps bytes 1:1 onto code points, so any byte string survives JSON
        ser_vocab = {k.decode("latin1"): v for k, v in self.encoder.items()}
        with open(vocab_path, "w", encoding="utf-8") as f:
            json.dump(ser_vocab, f, ensure_ascii=False, indent=2)

        # JSON (not a space-separated text file): tokens can contain spaces, newlines, \r, \xa0 ...
        ser_merges = [[p1.decode("latin1"), p2.decode("latin1")] for (p1, p2) in self.merges]  # in learned order
        with open(merges_path, "w", encoding="utf-8") as f:
            json.dump(ser_merges, f, ensure_ascii=False)

        config = {
            "tokenizer_class": "PureByteLevelBPETokenizer",
            "vocab_size": len(self.encoder),
            "special_tokens": self.special_tokens,
            "pad_token_id": self.pad_token_id,
            "bos_token_id": self.bos_token_id,
            "eos_token_id": self.eos_token_id
        }
        with open(config_path, "w", encoding="utf-8") as f:
            json.dump(config, f, indent=2)

        print(f"Tokenizer saved successfully to {save_dir}")

    @classmethod
    def load(cls, save_dir: str) -> "PureByteLevelBPETokenizer":
        tok = cls()
        with open(os.path.join(save_dir, "vocab.json"), "r", encoding="utf-8") as f:
            ser_vocab = json.load(f)
        tok.encoder = {k.encode("latin1"): v for k, v in ser_vocab.items()}
        tok.decoder = {v: k.encode("latin1") for k, v in ser_vocab.items()}

        tok.merges, tok.merge_ranks = {}, {}
        with open(os.path.join(save_dir, "merges.json"), "r", encoding="utf-8") as f:
            for rank, (s1, s2) in enumerate(json.load(f)):
                p1, p2 = s1.encode("latin1"), s2.encode("latin1")
                tok.merges[(p1, p2)] = p1 + p2
                tok.merge_ranks[(p1, p2)] = rank
        return tok


In [ ]:
%%writefile phase2_finetune.py
#!/usr/bin/env python3
"""Phase 2 legal NER: fine-tune the Phase-1 Llama LM (partial layers OR LoRA) on fold1 + fold2,
then predict fold3 and write prediction.json + submission.zip.

Needs next to this file:  llama_scratch.py (Config, LlamaLM, PRESETS)  and  bpe_tokenizer.py.

SMOKE TESTS (CPU, minutes; build a tiny fake checkpoint in the real format and run every stage + self-checks):
  python phase2_finetune.py --smoke --method partial --tok_dir ./tok_phase1 --data_dir ./data --out_dir ./out_smoke_partial
  python phase2_finetune.py --smoke --method lora    --tok_dir ./tok_phase1 --data_dir ./data --out_dir ./out_smoke_lora

MINI REAL RUN (GPU, real checkpoint, tiny data: checks ckpt/ALiBi/tokenizer match end to end):
  python phase2_finetune.py --ckpt model.pt --method lora --epochs 1 --max_docs 10 --max_windows 40 --val_fold 2 ...

EXPERIMENT (train fold1, validate on fold2):   --val_fold 2
FINAL RUN (fold1+fold2, no holdout):           omit --val_fold / --val_frac, fixed --epochs
"""
import argparse, bisect, copy, json, math, os, random, sys, time, zipfile
from dataclasses import asdict
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F

LABELS = ["APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT",
          "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"]
TAGS = ["O"] + [f"{p}-{l}" for l in LABELS for p in "BI"]
T2I = {t: i for i, t in enumerate(TAGS)}
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
FILES = {"fold1-text.json": "Phase_2_LNER/Train/fold1-text.json",
         "fold1-gold.json": "Phase_2_LNER/Train/fold1-gold.json",
         "fold2-text.json": "Phase_2_LNER/Train/fold2-text.json",
         "fold2-gold.json": "Phase_2_LNER/Train/fold2-gold.json",
         "fold3-text.json": "Phase_2_LNER/Test/fold3-text.json"}


def log(*a): print(time.strftime("[%H:%M:%S]"), *a, flush=True)


# ------------------------------------------------------------------ data
def get_data_paths(args):
    P = {}
    for name, remote in FILES.items():
        local = os.path.join(args.data_dir, name)
        if not os.path.exists(local) and args.download:
            from huggingface_hub import hf_hub_download
            local = hf_hub_download(HF_DATASET_REPO, remote, repo_type="dataset",
                                    token=os.environ.get("HF_TOKEN"))
        if not os.path.exists(local):
            sys.exit(f"Missing {local} (use --download or put the files in --data_dir)")
        P[name] = local
    return P


def make_tokenizer_fns(tok, PAT):
    def encode_offsets(text):
        ids, offs = [], []
        for m in PAT.finditer(text):
            w, b0 = m.group(), m.start()
            b2c = []
            for ci, ch in enumerate(w): b2c += [ci] * len(ch.encode("utf-8"))
            pos = 0
            for t in tok._encode_word(w):
                nb = len(tok.decoder[t])
                s_c = b2c[min(pos, len(b2c) - 1)]
                e_c = b2c[min(pos + nb, len(b2c)) - 1] + 1
                pos += nb
                while s_c < e_c and w[s_c].isspace(): s_c += 1
                ids.append(t); offs.append((b0 + s_c, b0 + e_c))
        return ids, offs
    return encode_offsets


def bio(offs, ents):
    y = [0] * len(offs); starts = [a for a, _ in offs]
    for s, e, l, *_ in sorted(ents):
        i = bisect.bisect_left(starts, s)
        while i > 0 and offs[i - 1][1] > s: i -= 1
        first = True
        while i < len(offs) and offs[i][0] < e:
            a, b = offs[i]
            if a < b and b > s:
                y[i] = T2I[("B-" if first else "I-") + l]; first = False
            i += 1
    return y


def decode_tags(tag_ids, offs):
    spans, cur = [], None
    for t, (a, b) in zip(tag_ids, offs):
        if a == b: continue
        tag = TAGS[t]
        if tag == "O":
            if cur: spans.append(cur); cur = None
        elif tag[0] == "B" or cur is None or cur[2] != tag[2:]:
            if cur: spans.append(cur)
            cur = [a, b, tag[2:]]
        else: cur[1] = b
    if cur: spans.append(cur)
    return spans

def refine(spans, text):
    out = []
    for s, e, l in spans:
        while e > s + 1:
            c = text[e - 1]
            if c in ",;:" or c.isspace(): e -= 1
            elif c == "." and e - 2 >= s and (text[e - 2] in ")]\"'" or text[e - 2].isdigit()): e -= 1
            else: break
        out.append((s, e, l))
    return out

# increasing stride and context legth to try if the result gets better
def windows(n, L=1024, stride=768):
    out, st = [], 0
    while True:
        out.append((st, min(st + L, n)))
        if st + L >= n: break
        st += stride
    return out


HDR = 128   # header tokens re-attached in front of every window after the first

def with_header(ids, a, b, y=None):
    if a == 0:
        return ids[a:b], (None if y is None else y[a:b]), 0
    h = ids[:HDR]
    return h + ids[a:b], (None if y is None else [-100] * len(h) + y[a:b]), len(h)


# ------------------------------------------------------------------ official-style scoring / validation
def validate_spans(x, where):
    """Mirrors evaluate_phase2.py format rules."""
    seen = set()
    for j, s in enumerate(x):
        assert isinstance(s, list) and len(s) in (3, 4), f"{where}[{j}] bad shape"
        a, b, l = s[:3]
        assert type(a) is int and type(b) is int and 0 <= a < b, f"{where}[{j}] bad offsets"
        assert l in LABELS, f"{where}[{j}] bad label {l!r}"
        assert len(s) == 3 or isinstance(s[3], str), f"{where}[{j}] bad text"
        assert (a, b, l) not in seen, f"{where}[{j}] duplicate"
        seen.add((a, b, l))


def strict_f1(preds, golds):
    per = {l: [0, 0, 0] for l in LABELS}
    for d, g in golds.items():
        P = {tuple(x[:3]) for x in preds.get(d, [])}; G = {tuple(x[:3]) for x in g}
        for l in LABELS:
            p = {x for x in P if x[2] == l}; q = {x for x in G if x[2] == l}; tp = len(p & q)
            per[l][0] += tp; per[l][1] += len(p) - tp; per[l][2] += len(q) - tp
    rows = {}
    for l, (tp, fp, fn) in per.items():
        p = tp / (tp + fp) if tp + fp else 0; r = tp / (tp + fn) if tp + fn else 0
        rows[l] = (p, r, 2 * p * r / (p + r) if p + r else 0)
    macro = tuple(float(np.mean([v[i] for v in rows.values()])) for i in range(3))
    return macro, rows


# ------------------------------------------------------------------ model
class NER(nn.Module):
    def __init__(s, base, n):
        super().__init__(); s.base = base; s.drop = nn.Dropout(0.1)
        s.rnn = nn.LSTM(base.cfg.d_model, 384, batch_first=True, bidirectional=True)
        s.head = nn.Linear(768, n); s.first_train = 0

    def hidden(s, idx, collect=False):
        b = s.base; T = idx.shape[1]; x = b.tok_emb(idx)
        # Phase-1 v2 uses ALiBi: rope_cos is None there, so branch on the checkpoint's own position scheme
        if b.cfg.pos_enc == "rope": cos, sin, slopes = b.rope_cos[:T], b.rope_sin[:T], None
        else: cos, sin, slopes = None, None, b.alibi_slopes
        hs = [x]
        for i, l in enumerate(b.layers):
            if i < s.first_train:
                with torch.no_grad(): x, _ = l(x, cos, sin, slopes, None, None, False)
            else: x, _ = l(x, cos, sin, slopes, None, None, False)
            if collect: hs.append(x)
        x = b.norm(x)
        return (x, hs) if collect else x

    def forward(s, idx, lengths=None):
        h = s.hidden(idx)
        with torch.autocast(device_type=h.device.type, enabled=False):
            h = h.float()
            if lengths is not None:
                pk = nn.utils.rnn.pack_padded_sequence(h, lengths.cpu(), batch_first=True, enforce_sorted=False)
                out, _ = s.rnn(pk)
                h, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=idx.shape[1])
            else:
                h, _ = s.rnn(h)
        return s.head(s.drop(h))

def autocast(dev):
    return torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=(dev.type == "cuda"))


# ------------------------------------------------------------------ LoRA
class LoRALinear(nn.Module):
    """y = W x + (alpha/r) * B(A(dropout(x))). W is frozen; B starts at 0 so step 0 == the base model."""
    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base, self.scale = base, alpha / r
        w = base.weight
        self.A = nn.Parameter(torch.empty(r, base.in_features, device=w.device, dtype=w.dtype))
        self.B = nn.Parameter(torch.zeros(base.out_features, r, device=w.device, dtype=w.dtype))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(self.drop(x), self.A), self.B) * self.scale

    @torch.no_grad()
    def merge(self):                       # fold the adapter into W (used by the smoke self-test only)
        self.base.weight += (self.B @ self.A) * self.scale
        self.B.zero_()


LORA_NAMES = {"wq", "wk", "wv", "wo", "w_gate", "w_up", "w_down"}


def apply_lora(base, targets, r, alpha, dropout, layer_ids=None):
    assert set(targets) <= LORA_NAMES, f"unknown lora target(s): {set(targets) - LORA_NAMES}"
    n = 0
    for i, blk in enumerate(base.layers):
        if layer_ids is not None and i not in layer_ids: continue
        for mod in (blk.attn, blk.mlp):
            for name in targets:
                lin = getattr(mod, name, None)
                if isinstance(lin, nn.Linear):
                    setattr(mod, name, LoRALinear(lin, r, alpha, dropout)); n += 1
    return n


def trainable_sd(model, names):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if k in names}


def load_trainable(model, sd):
    r = model.load_state_dict(sd, strict=False)
    assert not r.unexpected_keys, f"unexpected keys in checkpoint: {r.unexpected_keys[:3]}"


@torch.no_grad()
def lm_sanity(base, wins, dev, n=8):
    """Plain next-token loss of the Phase-1 LM on the labelled text. Wrong tokenizer/ckpt => loss >> 5."""
    base.eval(); tot, cnt = 0.0, 0
    for ids, _ in wins[::max(1, len(wins) // n)][:n]:
        if len(ids) < 16: continue
        x = torch.tensor([ids[:-1]], device=dev); y = torch.tensor([ids[1:]], device=dev)
        with autocast(dev): tot += base(x, targets=y).item() * y.numel()
        cnt += y.numel()
    return tot / max(cnt, 1)


def smoke_checks(model, args, probe, snap, names, saved_path):
    model.eval()
    for n_, p in model.named_parameters():
        if n_ in snap: assert torch.equal(p.detach(), snap[n_]), f"frozen weight changed: {n_}"
    log(f"check 1: {len(snap)} frozen tensors unchanged after training")
    saved = torch.load(saved_path, map_location="cpu")
    assert set(saved) == names, "saved checkpoint keys != trainable params"
    m3 = copy.deepcopy(model)
    with torch.no_grad():
        for n_, p in m3.named_parameters():
            if n_ in names: p.zero_()
    load_trainable(m3, saved); m3.eval()
    with torch.no_grad(): assert torch.allclose(model(probe).float(), m3(probe).float(), atol=1e-6), "save/load round trip differs"
    log(f"check 2: save/load round trip exact ({len(saved)} tensors, {sum(v.numel() for v in saved.values()) / 1e6:.2f}M params)")
    if args.method == "lora":
        Bs = [p for n_, p in model.named_parameters() if n_.endswith(".B")]
        assert Bs and any(float(p.abs().sum()) > 0 for p in Bs), "LoRA B never moved: adapters get no gradient"
        m2 = copy.deepcopy(model).eval()
        for m in m2.modules():
            if isinstance(m, LoRALinear): nn.init.normal_(m.B, std=0.02)    # make the delta clearly non-zero
        with torch.no_grad(): a = m2(probe).float()
        for m in m2.modules():
            if isinstance(m, LoRALinear): m.merge()
        with torch.no_grad(): b = m2(probe).float()
        assert torch.allclose(a, b, atol=1e-4), f"merge != adapter forward (max diff {(a - b).abs().max():.2e})"
        log("check 3: LoRA B moved during training; merged weights == adapter forward")


@torch.no_grad()
def layer_scores(model, wins, dev, seed, n=64):
    model.eval(); sims = []
    for ids, _ in random.Random(seed).sample(wins, min(n, len(wins))):
        x = torch.tensor([ids], device=dev)
        with autocast(dev): _, hs = model.hidden(x, collect=True)
        H = F.normalize(torch.stack([h[0].float().mean(0) for h in hs]), dim=-1)
        sims.append((H @ H.T).cpu())
    S = torch.stack(sims).mean(0); L = S.shape[0] - 1
    return ((S.sum(1) - 1) / L)[1:]


def select_block(score, k):
    L = len(score); k = min(k, L)
    s = max(range(L - k + 1), key=lambda i: score[i:i + k].sum().item())
    return list(range(s, s + k))


_allow = torch.ones(len(TAGS), len(TAGS), dtype=torch.bool)   # [prev, next]
for j, t in enumerate(TAGS):
    if t.startswith("I-"):
        for i, p in enumerate(TAGS):
            _allow[i, j] = (p != "O") and p[2:] == t[2:]
_start_pen = torch.tensor([-1e4 if t.startswith("I-") else 0.0 for t in TAGS])
_trans = (~_allow).float() * -1e4

def viterbi(logp):                       # logp: (n, K)
    n, K = logp.shape
    score = logp[0] + _start_pen
    back = torch.zeros(n, K, dtype=torch.long)
    for i in range(1, n):
        best, idx = (score[:, None] + _trans).max(0)
        back[i] = idx; score = best + logp[i]
    path = [int(score.argmax())]
    for i in range(n - 1, 0, -1): path.append(int(back[i][path[-1]]))
    return path[::-1]



@torch.no_grad()
def predict_doc(model, ids, offs, dev):
    model.eval(); n = len(ids)
    if n == 0: return []
    acc = torch.zeros(n, len(TAGS)); cnt = torch.zeros(n)
    for a, b in windows(n, L=1024 - HDR):
        w, _, off = with_header(ids, a, b)
        with autocast(dev):
            pr = model(torch.tensor([w], device=dev, dtype=torch.long))[0].float().softmax(-1).cpu()
        acc[a:b] += pr[off:]; cnt[a:b] += 1
    return decode_tags(viterbi((acc / cnt[:, None].clamp(min=1)).clamp(min=1e-9).log()), offs)

def evaluate(model, docs, keys, dev):
    preds = {k: refine(predict_doc(model, docs[k]["ids"], docs[k]["offs"], dev), docs[k]["text"]) for k in keys}
    return strict_f1(preds, {k: [tuple(e[:3]) for e in docs[k]["ents"]] for k in keys})

def error_breakdown(preds, golds):
    from collections import Counter, defaultdict
    out = defaultdict(Counter)
    for d, G in golds.items():
        P = [tuple(x[:3]) for x in preds.get(d, [])]; Ps = set(P)
        for s, e, l in set(G):
            if (s, e, l) in Ps: out[l]["exact"] += 1; continue
            ov = [p for p in P if p[0] < e and s < p[1]]
            if any(p[2] == l for p in ov): out[l]["boundary"] += 1
            elif ov: out[l]["wrong_label"] += 1
            else: out[l]["missed"] += 1
        for s, e, l in P:
            if not any(g[0] < e and s < g[1] for g in G): out[l]["spurious"] += 1
    return out



# ------------------------------------------------------------------ main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--smoke", action="store_true")
    ap.add_argument("--ckpt", default=None, help="Phase-1/2 LM checkpoint (final_model.pt); omitted in smoke => random tiny model")
    ap.add_argument("--tok_dir", required=True)
    ap.add_argument("--data_dir", default="./data")
    ap.add_argument("--download", action="store_true")
    ap.add_argument("--test_path", default=None, help="override fold3-text.json path")
    ap.add_argument("--out_dir", default="./out")
    ap.add_argument("--epochs", type=int, default=5)
    ap.add_argument("--bs", type=int, default=8)
    ap.add_argument("--lr_bb", type=float, default=5e-5)
    ap.add_argument("--lr_head", type=float, default=1e-3)
    ap.add_argument("--k_layers", type=int, default=4)
    ap.add_argument("--o_weight", type=float, default=0.3)
    ap.add_argument("--val_frac", type=float, default=0.0, help="0 = train on everything (final run)")
    ap.add_argument("--seed", type=int, default=67)
    ap.add_argument("--method", choices=["partial", "lora"], default="partial")
    ap.add_argument("--lora_r", type=int, default=16)
    ap.add_argument("--lora_alpha", type=float, default=32)
    ap.add_argument("--lora_dropout", type=float, default=0.05)
    ap.add_argument("--lora_targets", default="wq,wk,wv,wo,w_gate,w_up,w_down")
    ap.add_argument("--lora_layers", choices=["all", "sel"], default="all", help="sel = only the training-free selected block")
    ap.add_argument("--lr_lora", type=float, default=3e-4)
    ap.add_argument("--val_fold", type=int, default=0, choices=[0, 1, 2], help="hold out this WHOLE fold for validation (0 = off)")
    ap.add_argument("--max_docs", type=int, default=0, help="debug: first N docs per fold only (0 = all)")
    ap.add_argument("--max_windows", type=int, default=0, help="debug: cap training windows (0 = all)")
    ap.add_argument("--smoke_pos", choices=["alibi", "rope"], default="alibi", help="position scheme of the fake smoke checkpoint")
    args = ap.parse_args()

    if args.smoke:
        args.epochs = 2; args.bs = 4                 # 2 epochs so best/last checkpoint logic is exercised
        args.max_docs = args.max_docs or 8; args.max_windows = args.max_windows or 24
        if args.val_frac == 0 and not args.val_fold: args.val_frac = 0.3   # exercise the eval path too
    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    os.makedirs(args.out_dir, exist_ok=True)
    log(f"device={dev} smoke={args.smoke}")

    # tokenizer + model code (your own modules)
    sys.path.insert(0, os.getcwd())
    try:
        from llama_scratch import Config, LlamaLM, PRESETS
        from bpe_tokenizer import PureByteLevelBPETokenizer, _PAT
    except ImportError as e:
        sys.exit(f"Import failed: {e}. Put llama_scratch.py and bpe_tokenizer.py next to this script.")
    tok = PureByteLevelBPETokenizer.load(args.tok_dir)
    encode_offsets = make_tokenizer_fns(tok, _PAT)

    # LM
    if args.smoke and not args.ckpt:
        # fake Phase-1 checkpoint in the SAME format as the real one -> the real --ckpt code path is exercised
        tcfg = Config(vocab_size=33000, max_seq_len=512, rope_cache_len=1024, attn_impl="sdpa",
                      pos_enc=args.smoke_pos, **PRESETS["tiny"])
        args.ckpt = os.path.join(args.out_dir, "fake_phase1.pt")
        torch.save({"cfg": asdict(tcfg), "model": LlamaLM(tcfg).state_dict(), "step": 0, "val": 0.0}, args.ckpt)
        log(f"SMOKE: wrote tiny random fake checkpoint (pos_enc={args.smoke_pos}); scores will be ~0")
    if not args.ckpt: sys.exit("--ckpt is required for a real run")
    ck = torch.load(args.ckpt, map_location="cpu", weights_only=False)
    log(f"ckpt: step={ck.get('step')} val={ck.get('val')}")
    cfg = Config(**ck["cfg"]); cfg.attn_impl = "sdpa"
    base = LlamaLM(cfg); base.load_state_dict(ck["model"]); del ck
    log(f"LM: layers={cfg.n_layers} d_model={cfg.d_model} pos_enc={cfg.pos_enc} vocab={cfg.vocab_size}")

    # data
    P = get_data_paths(args)
    docs = {}
    max_docs = args.max_docs or None
    for fold in (1, 2):
        T = json.load(open(P[f"fold{fold}-text.json"], encoding="utf-8"))
        G = json.load(open(P[f"fold{fold}-gold.json"], encoding="utf-8"))
        for n, (d, t) in enumerate(T.items()):
            if max_docs and n >= max_docs: break
            ids, offs = encode_offsets(t)
            ents = G.get(d, [])
            docs[f"{fold}_{d}"] = dict(fold=fold, ids=ids, offs=offs, ents=ents, text=t, y=bio(offs, ents))
            if (n + 1) % 50 == 0: log(f"tokenized fold{fold}: {n + 1}/{len(T)} docs")
    keys = sorted(docs)
    assert max(len(docs[k]["ids"]) for k in keys) > 0
    assert max(max(d["ids"]) for d in docs.values()) < cfg.vocab_size, "token id >= model vocab: wrong tokenizer for this checkpoint"
    log(f"{len(docs)} labelled docs loaded")

    rng = random.Random(args.seed)
    val_keys = []
    if args.val_fold:
        val_keys = [k for k in keys if docs[k]["fold"] == args.val_fold]
    elif args.val_frac > 0:
        for f in (1, 2):
            ks = [k for k in keys if docs[k]["fold"] == f]; rng.shuffle(ks)
            val_keys += ks[:max(1, int(len(ks) * args.val_frac))]
    vs = set(val_keys); train_keys = [k for k in keys if k not in vs]
    log(f"train docs={len(train_keys)} val docs={len(val_keys)}")

    # tokenisation <-> gold alignment upper bound (should be ~1.0 on every doc)
    oracle = {k: decode_tags(docs[k]["y"], docs[k]["offs"]) for k in keys}
    (_, _, of1), orows = strict_f1(oracle, {k: [tuple(e[:3]) for e in docs[k]["ents"]] for k in keys})
    log(f"alignment upper-bound macro-F1 = {of1:.4f}")
    if not args.smoke and of1 < 0.97:
        log("WARNING: alignment bound < 0.97 - tokenizer offsets/BIO conversion is losing entities")

    train_win = []
    for k in train_keys:
        ids_, y_ = docs[k]["ids"], docs[k]["y"]
        for a, b in windows(len(ids_), L=min(1024, cfg.max_seq_len) - HDR, stride=768):
            w, yw, _ = with_header(ids_, a, b, y_)
            train_win.append((w, yw))
    if args.max_windows: train_win = train_win[:args.max_windows]
    log(f"{len(train_win)} training windows")
    assert train_win, "no training windows"

    # NER model
    model = NER(base, len(TAGS)).to(dev)
    if not args.smoke:
        lm_loss = lm_sanity(model.base, train_win, dev)
        log(f"LM sanity on labelled text: loss {lm_loss:.3f} (ppl {math.exp(lm_loss):.1f}); Phase-1 val loss was ~2-2.5")
        if lm_loss > 5: log("WARNING: loss > 5 -> tokenizer / checkpoint mismatch (or position-scheme bug). Do NOT train yet.")
    probe = torch.tensor([train_win[0][0][:64]], device=dev)
    model.eval()
    with torch.no_grad(): ref = model(probe).float()

    # training-free layer selection (always computed; used by 'partial', optionally by 'lora')
    score = layer_scores(model, train_win, dev, args.seed)
    SEL = select_block(score, args.k_layers)
    log(f"layer scores={[round(x, 3) for x in score.tolist()]} selected={SEL}")

    for p in model.parameters(): p.requires_grad = False
    if args.method == "partial":
        for i in SEL:
            for p in model.base.layers[i].parameters(): p.requires_grad = True
        for p in model.base.norm.parameters(): p.requires_grad = True
        model.first_train = min(SEL)
    else:
        lids = set(SEL) if args.lora_layers == "sel" else None
        n_ad = apply_lora(model.base, args.lora_targets.split(","), args.lora_r, args.lora_alpha, args.lora_dropout, lids)
        assert n_ad > 0, "no LoRA adapters were created"
        model.first_train = min(SEL) if lids else 0
        model.eval()
        with torch.no_grad(): out0 = model(probe).float()
        assert torch.allclose(ref, out0, atol=1e-5), "LoRA init changed the model output (B must start at 0)"
        log(f"LoRA: {n_ad} adapted matrices | r={args.lora_r} alpha={args.lora_alpha} targets={args.lora_targets} layers={args.lora_layers}")
    for p in model.head.parameters(): p.requires_grad = True
    for p in model.rnn.parameters(): p.requires_grad = True
    TRAIN_NAMES = {n_ for n_, p in model.named_parameters() if p.requires_grad}
    snap = {n_: p.detach().clone() for n_, p in model.named_parameters() if n_ not in TRAIN_NAMES} if args.smoke else {}
    lr_bb = args.lr_lora if args.method == "lora" else args.lr_bb
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    log(f"method={args.method} trainable {tr / 1e6:.2f}M / {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

    head_p = list(model.head.parameters()) + list(model.rnn.parameters())
    bb_p = [p for n_, p in model.named_parameters() if p.requires_grad and not n_.startswith(("head", "rnn"))]
    opt = torch.optim.AdamW([{"params": bb_p, "lr": lr_bb}, {"params": head_p, "lr": args.lr_head}], weight_decay=0.01)
    steps = args.epochs * math.ceil(len(train_win) / args.bs)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[lr_bb, args.lr_head], total_steps=steps, pct_start=0.1)
    scaler = torch.amp.GradScaler("cuda", enabled=(dev.type == "cuda"))
    w = torch.ones(len(TAGS), device=dev); w[0] = args.o_weight
    pad = tok.pad_token_id
    best, best_path, last_path = -1, os.path.join(args.out_dir, "ner_best.pt"), os.path.join(args.out_dir, "ner_last.pt")
    
    n_per_ep = math.ceil(len(train_win) / args.bs); t0 = time.time()
    for ep in range(args.epochs):
        model.train(); random.Random(args.seed + ep).shuffle(train_win); tot, nb = 0.0, 0
        for i in range(0, len(train_win), args.bs):
            b = train_win[i:i + args.bs]; L = max(len(x) for x, _ in b)
            X = torch.tensor([x + [pad] * (L - len(x)) for x, _ in b], device=dev)
            Y = torch.tensor([y + [-100] * (L - len(y)) for _, y in b], device=dev)
            lengths = torch.tensor([len(x) for x, _ in b])
            with autocast(dev):
                loss = F.cross_entropy(model(X, lengths).float().view(-1, len(TAGS)), Y.view(-1), weight=w, ignore_index=-100)
            assert torch.isfinite(loss), "non-finite loss"
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for g in opt.param_groups for p in g["params"]], 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item(); nb += 1
            if nb % 25 == 0 or nb == n_per_ep:
                done = ep * n_per_ep + nb; el = time.time() - t0
                log(f"ep {ep+1}/{args.epochs} step {nb}/{n_per_ep} loss {loss.item():.4f} | "
                    f"elapsed {el/60:.1f}m | ETA {el/done*(steps-done)/60:.1f}m")
        msg = f"epoch {ep + 1}/{args.epochs} mean loss {tot / nb:.4f}"
        if val_keys:
            (mp, mr, mf), rows = evaluate(model, docs, val_keys, dev)
            msg += f" | val strict macro P {mp:.3f} R {mr:.3f} F1 {mf:.3f}"
            if mf > best: best = mf; torch.save(trainable_sd(model, TRAIN_NAMES), best_path)
        log(msg)
        torch.save(trainable_sd(model, TRAIN_NAMES), last_path)

    if val_keys and os.path.exists(best_path):
        load_trainable(model, torch.load(best_path, map_location=dev))
        log("per-label val F1: " + str({l: round(v[2], 3) for l, v in evaluate(model, docs, val_keys, dev)[1].items()}))
        _p = {k: refine(predict_doc(model, docs[k]["ids"], docs[k]["offs"], dev), docs[k]["text"]) for k in val_keys}
        _g = {k: [tuple(e[:3]) for e in docs[k]["ents"]] for k in val_keys}
        for l, c in error_breakdown(_p, _g).items(): log(l, dict(c))
        for l in ("PREC", "CASENO", "STAT"):
            shown = 0
            for k in val_keys:
                t = docs[k]["text"]; G = [tuple(e[:3]) for e in docs[k]["ents"] if e[2] == l]
                for s, e, lab in _p[k]:
                    if lab != l or (s, e, l) in G: continue
                    g = next((x for x in G if x[0] < e and s < x[1]), None)
                    if g and shown < 8:
                        log(l, "PRED", repr(t[s:e][:80]), "| GOLD", repr(t[g[0]:g[1]][:80])); shown += 1
    else:
        log("no holdout: using final-epoch weights")

    final_path = best_path if (val_keys and os.path.exists(best_path)) else last_path
    if args.smoke: smoke_checks(model, args, probe, snap, TRAIN_NAMES, final_path)

    # predict fold3
    test_path = args.test_path or P["fold3-text.json"]
    T = json.load(open(test_path, encoding="utf-8"))
    lim = 2 if args.smoke else args.max_docs
    if lim: T = dict(list(T.items())[:lim])
    out = {}
    for n, (d, t) in enumerate(T.items()):
        if n % 50 == 0: log(f"predicting {n}/{len(T)}")
        ids, offs = encode_offsets(t)
        spans = {(s, e, l) for s, e, l in refine(predict_doc(model, ids, offs, dev), t) if 0 <= s < e <= len(t)}
        out[d] = [[s, e, l, t[s:e]] for s, e, l in sorted(spans)]
        validate_spans(out[d], f"pred[{d}]")
    assert set(out) == set(T), "id mismatch"
    pred_path = os.path.join(args.out_dir, "prediction.json")
    json.dump(out, open(pred_path, "w", encoding="utf-8"), ensure_ascii=False)
    zip_path = os.path.join(args.out_dir, "submission.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z: z.write(pred_path, arcname="prediction.json")
    json.dump({"args": vars(args), "selected_layers": SEL, "trainable_tensors": sorted(TRAIN_NAMES)[:5] + ["..."]},
              open(os.path.join(args.out_dir, "run_meta.json"), "w"), indent=1, default=str)
    # reload as strict JSON to make sure the file the grader sees is valid
    json.load(open(pred_path, encoding="utf-8"))
    log(f"OK: {len(out)} docs, {sum(len(v) for v in out.values())} entities -> {zip_path}")
    if args.smoke: log("SMOKE TEST PASSED (all stages ran end to end)")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile llama_scratch.py
"""
Llama-style decoder-only LM, written from scratch (no HF / no llama library).

Pieces: RMSNorm, RoPE (or ALiBi), grouped-query attention (GQA; MQA = n_kv_heads 1,
MHA = n_kv_heads == n_heads), SwiGLU MLP, pre-norm residual blocks, tied embeddings,
KV-cache generation with left-padded batches.

Only torch tensors/autograd/nn.Linear/nn.Embedding are used as building blocks.
"""
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


# ----------------------------------------------------------------------------- config
@dataclass
class Config:
    vocab_size: int = 33000
    d_model: int = 1024
    n_layers: int = 16
    n_heads: int = 16
    n_kv_heads: int = 4          # 16 query heads share 4 KV heads (4 queries per KV head)
    ffn_hidden: int = 2560
    max_seq_len: int = 1024      # training context
    rope_cache_len: int = 4096   # RoPE table length (>= prompt + generated tokens)
    rope_theta: float = 10000.0
    norm_eps: float = 1e-5
    tie_embeddings: bool = True
    pos_enc: str = "rope"        # "rope" | "alibi"
    attn_impl: str = "manual"    # "manual" (hand-written softmax(QK^T)V) | "sdpa" (torch fused kernel, same math)


PRESETS = {
    # exact totals with vocab 33000, tied embeddings
    "v2_100m": dict(d_model=768, n_layers=12, n_heads=12, n_kv_heads=4, ffn_hidden=2048), # ~100.86M
    "target_150m": dict(d_model=896, n_layers=14, n_heads=14, n_kv_heads=2, ffn_hidden=2304), # ~142M 
    "base":  dict(d_model=1024, n_layers=16, n_heads=16, n_kv_heads=4, ffn_hidden=2560),  # ~201.6M
    "small": dict(d_model=1024, n_layers=12, n_heads=16, n_kv_heads=4, ffn_hidden=2816),  # ~169.1M
    "tiny":  dict(d_model=64,   n_layers=2,  n_heads=4,  n_kv_heads=2, ffn_hidden=128),   # smoke tests
}


def analytic_param_count(c: Config) -> int:
    """Closed-form parameter count (use it to size a model before building it)."""
    hd = c.d_model // c.n_heads
    attn = 2 * c.d_model * c.n_heads * hd + 2 * c.d_model * c.n_kv_heads * hd   # Wq,Wo + Wk,Wv
    mlp = 3 * c.d_model * c.ffn_hidden                                           # gate, up, down
    per_layer = attn + mlp + 2 * c.d_model                                       # + 2 RMSNorm
    total = per_layer * c.n_layers + c.vocab_size * c.d_model + c.d_model        # + embedding + final norm
    if not c.tie_embeddings:
        total += c.vocab_size * c.d_model
    return total


def count_params(model, verbose=True) -> int:
    """model.parameters() yields a tied (shared) tensor only once, so tied weights are counted once."""
    total = sum(p.numel() for p in model.parameters())
    if verbose:
        emb = model.tok_emb.weight.numel()
        blocks = sum(p.numel() for p in model.layers.parameters())
        print(f"total {total/1e6:.1f}M | transformer blocks {blocks/1e6:.1f}M | embedding {emb/1e6:.1f}M "
              f"({'tied with output head' if model.cfg.tie_embeddings else 'output head separate'}) "
              f"| analytic {analytic_param_count(model.cfg)/1e6:.1f}M")
    return total


# ----------------------------------------------------------------------------- RMSNorm
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()                                   # always normalise in fp32
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight.float()).type_as(x)


# ----------------------------------------------------------------------------- RoPE
def build_rope_cache(head_dim, max_len, theta):
    inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))   # (hd/2,)
    t = torch.arange(max_len).float()
    freqs = torch.outer(t, inv_freq)                                                # (max_len, hd/2)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    """x: (B, H, T, hd); cos/sin: (T, hd/2). Rotates the two halves of each head vector (Llama layout)."""
    half = x.shape[-1] // 2
    xf = x.float()
    x1, x2 = xf[..., :half], xf[..., half:]
    cos = cos[None, None]
    sin = sin[None, None]
    out = torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return out.type_as(x)


# ----------------------------------------------------------------------------- ALiBi
def get_alibi_slopes(n_heads):
    def pow2(n):
        start = 2 ** (-8.0 / n)
        return [start ** (i + 1) for i in range(n)]
    if math.log2(n_heads).is_integer():
        s = pow2(n_heads)
    else:
        c = 2 ** math.floor(math.log2(n_heads))
        s = pow2(c) + pow2(2 * c)[0::2][: n_heads - c]
    return torch.tensor(s, dtype=torch.float32)


# ----------------------------------------------------------------------------- GQA
def repeat_kv(x, n_rep):
    """(B, n_kv, S, hd) -> (B, n_kv*n_rep, S, hd); query head h uses kv head h // n_rep."""
    if n_rep == 1:
        return x
    B, n_kv, S, hd = x.shape
    return x[:, :, None].expand(B, n_kv, n_rep, S, hd).reshape(B, n_kv * n_rep, S, hd)


class GQAttention(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0 and cfg.n_heads % cfg.n_kv_heads == 0
        self.nh, self.nkv = cfg.n_heads, cfg.n_kv_heads
        self.hd = cfg.d_model // cfg.n_heads
        self.n_rep = self.nh // self.nkv
        self.impl = cfg.attn_impl
        self.wq = nn.Linear(cfg.d_model, self.nh * self.hd, bias=False)
        self.wk = nn.Linear(cfg.d_model, self.nkv * self.hd, bias=False)
        self.wv = nn.Linear(cfg.d_model, self.nkv * self.hd, bias=False)
        self.wo = nn.Linear(self.nh * self.hd, cfg.d_model, bias=False)

    def forward(self, x, cos, sin, alibi_slopes, key_valid, past, use_cache):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.nh, self.hd).transpose(1, 2)    # (B, nh,  T, hd)
        k = self.wk(x).view(B, T, self.nkv, self.hd).transpose(1, 2)   # (B, nkv, T, hd)
        v = self.wv(x).view(B, T, self.nkv, self.hd).transpose(1, 2)

        if cos is not None:                                            # RoPE on q and k (not v)
            q = apply_rope(q, cos, sin)
            k = apply_rope(k, cos, sin)

        if past is not None:                                           # cache holds only nkv heads
            k = torch.cat([past[0], k], dim=2)
            v = torch.cat([past[1], v], dim=2)
        new_cache = (k, v) if use_cache else None
        S = k.shape[2]

        k = repeat_kv(k, self.n_rep)
        v = repeat_kv(v, self.n_rep)

        # relative distance of every (query, key) pair; keys j <= query position are visible
        qpos = torch.arange(S - T, S, device=x.device)
        kpos = torch.arange(S, device=x.device)
        rel = qpos[:, None] - kpos[None, :]                            # (T, S)
        allowed = (rel >= 0)[None, None]                               # (1, 1, T, S)
        if key_valid is not None:                                      # hide left-padding keys
            allowed = allowed & key_valid[:, None, None, :]
        bias = None
        if alibi_slopes is not None:
            bias = -alibi_slopes.view(1, -1, 1, 1) * rel.clamp(min=0).float()[None, None]   # (1, nh, T, S)

        if self.impl == "sdpa" and key_valid is None and T == S:      # training fast path
            if bias is None:
                out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
            else:
                m = bias.masked_fill(~allowed, torch.finfo(q.dtype).min).to(q.dtype)
                out = F.scaled_dot_product_attention(q, k, v, attn_mask=m)
        else:                                                          # hand-written attention
            scores = ((q * (self.hd ** -0.5)) @ k.transpose(-2, -1)).float()   # (B, nh, T, S)
            if bias is not None:
                scores = scores + bias
            scores = scores.masked_fill(~allowed, -1e30)               # finite, so a fully-masked pad row can't give NaN
            probs = torch.softmax(scores, dim=-1).to(v.dtype)
            out = probs @ v                                            # (B, nh, T, hd)

        out = out.transpose(1, 2).reshape(B, T, self.nh * self.hd)
        return self.wo(out), new_cache


# ----------------------------------------------------------------------------- SwiGLU
class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden):
        super().__init__()
        self.w_gate = nn.Linear(d_model, hidden, bias=False)
        self.w_up = nn.Linear(d_model, hidden, bias=False)
        self.w_down = nn.Linear(hidden, d_model, bias=False)

    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))


# ----------------------------------------------------------------------------- block + model
class Block(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = GQAttention(cfg)
        self.mlp_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.mlp = SwiGLU(cfg.d_model, cfg.ffn_hidden)

    def forward(self, x, cos, sin, alibi_slopes, key_valid, past, use_cache):
        h, new_cache = self.attn(self.attn_norm(x), cos, sin, alibi_slopes, key_valid, past, use_cache)
        x = x + h
        x = x + self.mlp(self.mlp_norm(x))
        return x, new_cache


class LlamaLM(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        assert cfg.pos_enc in ("rope", "alibi") and cfg.attn_impl in ("manual", "sdpa")
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.layers = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.tok_emb.weight

        if cfg.pos_enc == "rope":
            cos, sin = build_rope_cache(cfg.d_model // cfg.n_heads, cfg.rope_cache_len, cfg.rope_theta)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)
            self.alibi_slopes = None
        else:
            self.rope_cos = self.rope_sin = None
            self.register_buffer("alibi_slopes", get_alibi_slopes(cfg.n_heads), persistent=False)

        self.apply(self._init_weights)
        for name, p in self.named_parameters():       # GPT-2 style: shrink the residual-branch output projections
            if name.endswith("wo.weight") or name.endswith("w_down.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layers))

    @staticmethod
    def _init_weights(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, use_cache=False, cache=None, start_pos=0,
                key_valid=None, all_logits=False):
        """
        training:  model(x, targets=y) -> scalar loss
        inference: model(idx, use_cache=True, cache=..., start_pos=..., key_valid=...) -> (logits, cache)
                   logits are for the last position only unless all_logits=True.
        """
        B, T = idx.shape
        x = self.tok_emb(idx)
        cos = sin = None
        if self.cfg.pos_enc == "rope":
            assert start_pos + T <= self.cfg.rope_cache_len, "sequence longer than the RoPE table"
            cos = self.rope_cos[start_pos:start_pos + T]
            sin = self.rope_sin[start_pos:start_pos + T]

        new_caches = []
        for i, layer in enumerate(self.layers):
            past = cache[i] if cache is not None else None
            x, c = layer(x, cos, sin, self.alibi_slopes, key_valid, past, use_cache)
            new_caches.append(c)
        x = self.norm(x)

        if targets is not None:
            logits = self.lm_head(x)
            return F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1), ignore_index=-100)
        logits = self.lm_head(x if all_logits else x[:, -1:, :])
        return logits, (new_caches if use_cache else None)


# ----------------------------------------------------------------------------- generation
@torch.no_grad()
def generate(model, prompts, max_new_tokens, eos_id, pad_id, temperature=0.1, top_k=30,
             top_p=0.95, rep_penalty=1.2, ban_eos=True):
    """
    prompts: list of lists of token ids (any lengths). Left-pads them into one batch, runs one prefill,
    then decodes with a KV cache. Returns a list of lists of generated ids (EOS / padding removed).
    temperature <= 0 -> greedy.
    """
    model.eval()
    device = next(model.parameters()).device
    V = model.cfg.vocab_size
    B = len(prompts)
    L = max(len(p) for p in prompts)
    ids = torch.full((B, L), pad_id, dtype=torch.long, device=device)
    key_valid = torch.zeros(B, L, dtype=torch.bool, device=device)
    seen = torch.zeros(B, V, dtype=torch.bool, device=device)
    for b, p in enumerate(prompts):
        t = torch.tensor(p, dtype=torch.long, device=device)
        ids[b, L - len(p):] = t
        key_valid[b, L - len(p):] = True
        seen[b, t] = True

    out = [[] for _ in range(B)]
    finished = torch.zeros(B, dtype=torch.bool, device=device)
    amp = torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == "cuda"))
    with amp:
        logits, cache = model(ids, use_cache=True, key_valid=key_valid, start_pos=0)
        for step in range(max_new_tokens):
            lg = logits[:, -1, :].float()
            if ban_eos:
                lg[:, eos_id] = float("-inf")
            if rep_penalty != 1.0:                                     # CTRL-style repetition penalty
                pen = torch.where(lg > 0, lg / rep_penalty, lg * rep_penalty)
                lg = torch.where(seen, pen, lg)
            if temperature <= 0:
                nxt = lg.argmax(-1)
            else:
                lg = lg / temperature
                if top_k and top_k > 0:
                    kth = torch.topk(lg, min(top_k, V)).values[:, -1:]
                    lg = lg.masked_fill(lg < kth, float("-inf"))
                if top_p < 1.0:
                    sl, si = torch.sort(lg, descending=True, dim=-1)
                    pr = torch.softmax(sl, dim=-1)
                    drop = (pr.cumsum(-1) - pr) > top_p                # keep the token that crosses top_p
                    sl = sl.masked_fill(drop, float("-inf"))
                    lg = torch.full_like(lg, float("-inf")).scatter(1, si, sl)
                nxt = torch.multinomial(torch.softmax(lg, dim=-1), 1).squeeze(1)

            nxt = torch.where(finished, torch.full_like(nxt, pad_id), nxt)
            for b in range(B):
                if not finished[b] and nxt[b].item() != eos_id:
                    out[b].append(int(nxt[b].item()))
            finished = finished | (nxt == eos_id)
            seen.scatter_(1, nxt[:, None], True)
            if bool(finished.all()):
                break
            key_valid = torch.cat([key_valid, torch.ones(B, 1, dtype=torch.bool, device=device)], dim=1)
            logits, cache = model(nxt[:, None], use_cache=True, cache=cache,
                                  start_pos=L + step, key_valid=key_valid)
    return out

In [ ]:
from huggingface_hub import hf_hub_download
for f in ["vocab.json","merges.json","tokenizer_config.json"]:
    hf_hub_download("DumbLad/legal-bpe-tokenizer-phase1-v2", f, repo_type="model",
                    local_dir="/kaggle/working/tok_phase1", token=HF_TOKEN)

In [ ]:
import shutil, os
os.makedirs("data", exist_ok=True)
for p in list(P.values()) + [TEST]:
    shutil.copy(p, "data/")
print(sorted(os.listdir("data")))

In [ ]:
import json, sys, torch
import importlib, phase2_finetune
importlib.reload(phase2_finetune)
sys.path.insert(0, ".")
from bpe_tokenizer import PureByteLevelBPETokenizer, _PAT
from phase2_finetune import *
tok = PureByteLevelBPETokenizer.load("/kaggle/working/tok_phase1")
enc = make_tokenizer_fns(tok, _PAT)
DOCS = {}
for f in (1, 2):
    T_ = json.load(open(f"data/fold{f}-text.json")); G_ = json.load(open(f"data/fold{f}-gold.json"))
    for d, t in T_.items(): DOCS[f"{f}_{d}"] = (t, G_.get(d, []))

In [ ]:
split = 0; lost = []
for k, (t, G) in DOCS.items():
    ids, offs = enc(t)
    assert len(ids) == len(offs) and tok.decode(ids) == t, k
    prev = 0
    for a, b in offs:
        assert 0 <= a <= b <= len(t), (k, a, b)
        if a < b: assert not t[a:b].isspace()
        assert a >= prev or not t[a].isascii(), (k, "ASCII overlap", a)
        prev = max(prev, b)
    for e in G:
        s_, e_, l_ = e[:3]
        assert 0 <= s_ < e_ <= len(t)
        if len(e) > 3: assert t[s_:e_] == e[3], (k, s_, e_, t[s_:e_], e[3])   # test 3
    got = {tuple(x) for x in decode_tags(bio(offs, G), offs)}
    for s_, e_, l_ in {tuple(e[:3]) for e in G} - got:                       # test 2
        lost.append((k, s_, e_, l_, t[max(0, s_-10):e_+10]))
print("entities unreachable by tokenization:", len(lost))
for x in lost[:30]: print(x)
print("max code point:", max(ord(c) for t, _ in DOCS.values() for c in t))
print("CRLF docs:", sum('\r' in t for t, _ in DOCS.values()))

In [ ]:
'''152 entities are unreachable.
Trailing punctuation merged into one token
boundary in the middle of a word. I don't think if I should do anyth to fix this one, I'll try fixing the first problem'''

In [ ]:
'''class Oracle:                       # stands in for the model, returns gold tags window by window
    def __init__(s, y): s.y = y; s.w = iter(windows(len(y)))
    def eval(s): pass
    def __call__(s, x):
        a, b = next(s.w); n = b - a
        out = torch.full((1, n, len(TAGS)), -20.0)
        out[0, torch.arange(n), torch.tensor(s.y[a:b])] = 20.0
        return out

bad = 0
for k, (t, G) in DOCS.items():
    ids, offs = enc(t); y = bio(offs, G)
    want = decode_tags(y, offs)
    lp = torch.full((len(y), len(TAGS)), -20.0); lp[torch.arange(len(y)), torch.tensor(y)] = 0
    assert [tuple(x) for x in decode_tags(viterbi(lp), offs)] == [tuple(x) for x in want], ("viterbi", k)   # test 4
    got = predict_doc(Oracle(y), ids, offs, torch.device("cpu"))                                            # test 5
    assert [tuple(x) for x in got] == [tuple(x) for x in want], ("windows", k); bad += 0
print("tests 4-5 OK on", len(DOCS), "docs; multi-window docs:", sum(len(enc(t)[0]) > 1024 for t, _ in DOCS.values()))'''

In [ ]:
def oracle(use_refine):
    preds, golds = {}, {}
    for k, (t, G) in DOCS.items():
        ids, offs = enc(t)
        sp = decode_tags(bio(offs, G), offs)
        preds[k] = [tuple(x) for x in (refine(sp, t) if use_refine else sp)]
        golds[k] = [tuple(e[:3]) for e in G]
    return strict_f1(preds, golds), preds, golds
for r in (False, True):
    print("refine", r, "oracle macro F1 %.4f" % oracle(r)[0][0][2])
_, P_, G_ = oracle(True)
miss = [(k, x) for k in G_ for x in set(G_[k]) - set(P_[k])]
print("still unreachable:", len(miss))
for k, (s, e, l) in miss[:20]: print(k, l, repr(DOCS[k][0][max(0, s-5):e+5]))

In [ ]:
'''import subprocess, re
CK = "/kaggle/input/datasets/koooshiieee/phase2v2/model.pt"
base = f"python phase2_finetune.py --ckpt {CK} --tok_dir /kaggle/working/tok_phase1 --data_dir ./data --epochs 8 --val_fold 2"
configs = {
 "A_lora_o0.6_r32": "--method lora --o_weight 0.6 --lora_r 32 --lr_lora 5e-4",
 "B_lora_o0.3_r32": "--method lora --o_weight 0.3 --lora_r 32 --lr_lora 5e-4",
}
for name, extra in configs.items():
    out = subprocess.run(f"{base} {extra} --out_dir /kaggle/working/out_{name}",
                         shell=True, capture_output=True, text=True)
    lines = [l for l in (out.stdout + out.stderr).splitlines() if "val strict" in l or "per-label" in l or "Error" in l]
    print("=====", name); print("\n".join(lines))

'''

===== A_lora_o0.6_r32 

[17:45:33] epoch 1/8 mean loss 1.1902 | val strict macro P 0.351 R 0.393 F1 0.362

[17:46:17] epoch 2/8 mean loss 0.1059 | val strict macro P 0.439 R 0.465 F1 0.440

[17:47:02] epoch 3/8 mean loss 0.0541 | val strict macro P 0.441 R 0.396 F1 0.400

[17:47:46] epoch 4/8 mean loss 0.0265 | val strict macro P 0.446 R 0.435 F1 0.428

[17:48:31] epoch 5/8 mean loss 0.0129 | val strict macro P 0.453 R 0.463 F1 0.447

[17:49:15] epoch 6/8 mean loss 0.0065 | val strict macro P 0.444 R 0.434 F1 0.428

[17:49:59] epoch 7/8 mean loss 0.0035 | val strict macro P 0.446 R 0.437 F1 0.429

[17:50:43] epoch 8/8 mean loss 0.0030 | val strict macro P 0.444 R 0.436 F1 0.428

[17:51:08] per-label val F1: {'APP': 0.161, 'RESP': 0.183, 'A.COUNSEL': 0.413, 'R.COUNSEL': 0.474, 'JUDGE': 0.554, 'WIT': 0.158, 'AUTH': 0.284, 'COURT': 0.796, 'STAT': 0.653, 'PREC': 0.392, 'DATE': 0.917, 'CASENO': 0.375}

===== B_lora_o0.3_r32

[17:52:35] epoch 1/8 mean loss 1.2855 | val strict macro P 0.389 R 0.446 F1 0.393

[17:53:19] epoch 2/8 mean loss 0.1390 | val strict macro P 0.425 R 0.495 F1 0.448

[17:54:04] epoch 3/8 mean loss 0.0663 | val strict macro P 0.432 R 0.428 F1 0.414

[17:54:48] epoch 4/8 mean loss 0.0333 | val strict macro P 0.448 R 0.468 F1 0.448

[17:55:33] epoch 5/8 mean loss 0.0177 | val strict macro P 0.456 R 0.479 F1 0.458

[17:56:17] epoch 6/8 mean loss 0.0095 | val strict macro P 0.449 R 0.449 F1 0.438

[17:57:02] epoch 7/8 mean loss 0.0053 | val strict macro P 0.453 R 0.445 F1 0.436

[17:57:46] epoch 8/8 mean loss 0.0044 | val strict macro P 0.452 R 0.442 F1 0.434

[17:58:11] per-label val F1: {'APP': 0.169, 'RESP': 0.15, 'A.COUNSEL': 0.48, 'R.COUNSEL': 0.475, 'JUDGE': 0.565, 'WIT': 0.221, 'AUTH': 0.294, 'COURT': 0.786, 'STAT': 
0.673, 'PREC': 0.392, 'DATE': 0.92, 'CASENO': 0.371}

In [ ]:
'''!python phase2_finetune.py --method lora --o_weight 0.3 --lora_r 16 --epochs 1 --bs 4 --max_docs 5 --max_windows 20 --val_fold 2 --ckpt /kaggle/input/datasets/koooshiieee/phase2v2/model.pt --tok_dir /kaggle/working/tok_phase1 --data_dir ./data --out_dir /kaggle/working/out_mini'''

In [ ]:
!python phase2_finetune.py --method lora --o_weight 0.3 --lora_r 16 --epochs 5 --bs 4 --ckpt /kaggle/input/datasets/koooshiieee/phase2v2/model.pt --tok_dir /kaggle/working/tok_phase1 --data_dir ./data --out_dir /kaggle/working/out_final

In [ ]:
!python phase2_finetune.py --method lora --o_weight 0.3 --lora_r 16 --epochs 5 --bs 4 --val_fold 2 --ckpt /kaggle/input/datasets/koooshiieee/phase2v2/model.pt --tok_dir /kaggle/working/tok_phase1 --data_dir ./data --out_dir /kaggle/working/out_val

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
repo = f"{api.whoami()['name']}/legal-ner-phase2-lora-v1"
RUN = "phase2v5_bidir_w1024_lora_r16_o0.3_e5"
api.create_repo(repo, repo_type="model", private=True, exist_ok=True)
api.upload_folder(folder_path="/kaggle/working/out_final", repo_id=repo, repo_type="model",
                  path_in_repo=RUN)
api.upload_folder(folder_path="/kaggle/working/out_val", repo_id=repo, repo_type="model",
                  path_in_repo=f"{RUN}/val")
api.upload_folder(folder_path="/kaggle/working/tok_phase1", repo_id=repo, repo_type="model",
                  path_in_repo=f"{RUN}/tokenizer")
print(f"https://huggingface.co/{repo}/tree/main/{RUN}")